# Kon-Tiki Biochar Volume — Video → Litres  (Colab GPU app)

**How to use:**
1. `Runtime → Change runtime type → GPU (T4)`
2. Run **Cell 1 (Setup)** once — installs everything and loads the model.
3. Run **Cell 2 (Measure a kiln)** — upload a slow-orbit video → get the volume.
   Re-run Cell 2 for each new kiln (Setup stays loaded).

> Reconstruction quality depends on capture — follow the video SOP (slow full circle,
> tilt ~50–60° down into the kiln, full rim always visible, 1080p+). The volume maths is
> validated to ~2–4% on ground truth; a proper video + a known-volume kiln proves the rest.


### Cell 1 · Setup — run once

In [ ]:
import os, sys, shutil, glob, base64, torch, numpy as np, cv2
# deps (keep Colab's matched torch/torchvision/numpy -> avoids nms/numpy breakage)
!pip -q install opencv-python-headless scipy 2>/dev/null
if not os.path.exists('vggt'):
    !git clone -q https://github.com/facebookresearch/vggt.git
!grep -viE '^(torch|torchvision|torchaudio|numpy)' vggt/requirements.txt > /tmp/r.txt
!pip -q install -r /tmp/r.txt 2>/dev/null
if 'vggt' not in sys.path: sys.path.append('vggt')
assert torch.cuda.is_available(), "No GPU. Runtime > Change runtime type > GPU (T4), then re-run."

# the tested volume engine (embedded, byte-identical to estimate_volume.py)
open("estimate_volume.py", "w", encoding="utf-8").write(base64.b64decode("IiIiVmlkZW8gLT4gVm9sdW1lIHBpcGVsaW5lLCBWT0xVTUUgU1RFUCAobG9jYWwsIENQVSDigJQgbm8gR1BVIG5lZWRlZCkuCgpUYWtlcyBhIDMtRCBwb2ludCBjbG91ZCBvZiBhIGJpb2NoYXItZmlsbGVkIEtvbi1UaWtpIGtpbG4gKGZyb20gdGhlIHJlY29uc3RydWN0aW9uCnN0ZXApIGFuZCByZXR1cm5zIHRoZSBiaW9jaGFyIHZvbHVtZSBpbiBsaXRyZXMuIFB1cmUgZ2VvbWV0cnk6CiAgMS4gZmluZCAndXAnIGZyb20gdGhlIGRvbWluYW50IHBsYW5lOyBwdXQgdGhlICp3aWRlc3QqIGhvcml6b250YWwgc2hlZXQgKGdyb3VuZCkKICAgICBhdCB0aGUgYm90dG9tICAoc28gd2UgbmV2ZXIgY29uZnVzZSB0aGUgYmlvY2hhciBzdXJmYWNlIGZvciB0aGUgZ3JvdW5kKSwKICAyLiBpc29sYXRlIHRoZSBraWxuLCBmaXQgdGhlIHJpbSAtPiBzY2FsZSB0aGUgY2xvdWQgdG8gcmVhbCBjbSAocmltIHJhZGl1cyA3NSBjbSksCiAgMy4gaW50ZWdyYXRlIHRoZSBtZWFzdXJlZCBiaW9jaGFyIHN1cmZhY2UgYWdhaW5zdCB0aGUga25vd24ga2lsbiBjb25lLCBmaWxsaW5nCiAgICAgZ2FwcyBieSBuZWFyZXN0LW5laWdoYm91ciBzbyBzcGFyc2Ugc3BvdHMgZG9uJ3QgdW5kZXItY291bnQuCgpTYW1lIGxvZ2ljIHRoZSBDb2xhYiBub3RlYm9vayB1c2VzOyBpdCBydW5zIGhlcmUgb24gQ1BVIGJlY2F1c2UgaXQgaXMgbm90IEdQVSB3b3JrLgoKVXNhZ2U6ICBweXRob24gZXN0aW1hdGVfdm9sdW1lLnB5IGNsb3VkLnBseSBbcmltX3JhZGl1c19jbV0gW3ZpZXdzLnBuZ10KIiIiCmltcG9ydCBzeXMsIG51bXB5IGFzIG5wCmZyb20gc2NpcHkuaW50ZXJwb2xhdGUgaW1wb3J0IE5lYXJlc3ROREludGVycG9sYXRvcgpmcm9tIHNjaXB5LnNwYXRpYWwgaW1wb3J0IGNLRFRyZWUKCiMgS29uLVRpa2kgMTAwMCBnZW9tZXRyeSAoY20pLCBmcm9tIHRoZSBkZXNpZ24gZHJhd2luZwpSX0NNLCBSQl9DTSwgSF9DTSA9IDc1LjAsIDQxLjE1LCA5My4wICAgIyByaW0gw5gxNTAwLCBib3R0b20gw5g4MjMsIGRlcHRoIDkzMCAoZGVzaWduIGRyYXdpbmcpCkRFTlNJVFkgPSAwLjI1ICAjIGtnIC8gTApDRUxMID0gMy4wICAgICAgIyBpbnRlZ3JhdGlvbiBncmlkIChjbSkKVE9QX1BDVCA9IDEyICAgICMgcGVyLWNlbGwgcGVyY2VudGlsZSA9IHRoZSB0b3AgKGJpb2NoYXIpIHN1cmZhY2UsIHJvYnVzdCB0byBkZWVwIGFydGVmYWN0cwpDT0xfTUlOID0gOS4wICAgIyBjbTogbWluIGJpb2NoYXIgY29sdW1uIHRvIGNvdW50IChyZWplY3RzIHRoZSBzdGVlcC13YWxsIHJpbmc7IH5DRUxMKkgvKFItUkIpKQoKCmRlZiByb3RfZnJvbV90byhhLCBiKToKICAgIGEgPSBhIC8gbnAubGluYWxnLm5vcm0oYSk7IGIgPSBiIC8gbnAubGluYWxnLm5vcm0oYikKICAgIHYgPSBucC5jcm9zcyhhLCBiKTsgYyA9IGZsb2F0KG5wLmRvdChhLCBiKSkKICAgIGlmIG5wLmxpbmFsZy5ub3JtKHYpIDwgMWUtODoKICAgICAgICByZXR1cm4gbnAuZXllKDMpIGlmIGMgPiAwIGVsc2UgbnAuZGlhZyhbMS4wLCAtMS4wLCAtMS4wXSkKICAgIHZ4ID0gbnAuYXJyYXkoW1swLCAtdlsyXSwgdlsxXV0sIFt2WzJdLCAwLCAtdlswXV0sIFstdlsxXSwgdlswXSwgMF1dKQogICAgcmV0dXJuIG5wLmV5ZSgzKSArIHZ4ICsgdnggQCB2eCAqICgxLjAgLyAoMS4wICsgYykpCgoKZGVmIGZpdF9jaXJjbGUoeHkpOgogICAgeCwgeSA9IHh5WzosIDBdLCB4eVs6LCAxXQogICAgQSA9IG5wLmNfWzIgKiB4LCAyICogeSwgbnAub25lcyhsZW4oeCkpXTsgYiA9IHggKiogMiArIHkgKiogMgogICAgYywgKl8gPSBucC5saW5hbGcubHN0c3EoQSwgYiwgcmNvbmQ9Tm9uZSkKICAgIGN4LCBjeSA9IGNbMF0sIGNbMV0KICAgIHJldHVybiBjeCwgY3ksIG5wLnNxcnQobWF4KGNbMl0gKyBjeCAqKiAyICsgY3kgKiogMiwgMWUtOSkpCgoKZGVmIF9zZWdtZW50X3BsYW5lKFAsIHRociwgaXRlcnM9MjAwMCwgc2VlZD0wKToKICAgICIiIk1pbmltYWwgUkFOU0FDIHBsYW5lIGZpdCAtPiAobm9ybWFsLCBpbmxpZXJfbWFzaykuIE5vIG9wZW4zZCBkZXBlbmRlbmN5LiIiIgogICAgcm5nID0gbnAucmFuZG9tLmRlZmF1bHRfcm5nKHNlZWQpCiAgICBiZXN0X24sIGJlc3RfaW4gPSBOb25lLCBOb25lCiAgICBuX2Jlc3QgPSAwCiAgICBmb3IgXyBpbiByYW5nZShpdGVycyk6CiAgICAgICAgaWR4ID0gcm5nLmNob2ljZShsZW4oUCksIDMsIHJlcGxhY2U9RmFsc2UpCiAgICAgICAgcDAsIHAxLCBwMiA9IFBbaWR4XQogICAgICAgIG5ybSA9IG5wLmNyb3NzKHAxIC0gcDAsIHAyIC0gcDApCiAgICAgICAgbmwgPSBucC5saW5hbGcubm9ybShucm0pCiAgICAgICAgaWYgbmwgPCAxZS05OgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIG5ybSA9IG5ybSAvIG5sCiAgICAgICAgZCA9IG5wLmFicygoUCAtIHAwKSBAIG5ybSkKICAgICAgICBpbmwgPSBkIDwgdGhyCiAgICAgICAgYyA9IGludChpbmwuc3VtKCkpCiAgICAgICAgaWYgYyA+IG5fYmVzdDoKICAgICAgICAgICAgbl9iZXN0LCBiZXN0X24sIGJlc3RfaW4gPSBjLCBucm0sIGlubAogICAgcmV0dXJuIGJlc3RfbiwgYmVzdF9pbgoKCmRlZiBfbGFyZ2VzdF9jbHVzdGVyKFAsIGVwcywgbWluX3B0cz0yMCk6CiAgICAiIiJHcmlkLWJhc2VkIGNvbm5lY3RlZC1jb21wb25lbnRzIGNsdXN0ZXJpbmcgKGZhc3QsIG5vIG9wZW4zZCkuIiIiCiAgICBrZXlzID0gbnAuZmxvb3IoUCAvIGVwcykuYXN0eXBlKG5wLmludDY0KQogICAgZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgZGVmYXVsdGRpY3QKICAgIGNlbGwgPSBkZWZhdWx0ZGljdChsaXN0KQogICAgZm9yIGksIGsgaW4gZW51bWVyYXRlKG1hcCh0dXBsZSwga2V5cykpOgogICAgICAgIGNlbGxba10uYXBwZW5kKGkpCiAgICBzZWVuLCBiZXN0ID0gc2V0KCksIFtdCiAgICBuZWlnaCA9IFsoZHgsIGR5LCBkeikgZm9yIGR4IGluICgtMSwgMCwgMSkgZm9yIGR5IGluICgtMSwgMCwgMSkgZm9yIGR6IGluICgtMSwgMCwgMSldCiAgICBmb3Igc3RhcnQgaW4gY2VsbDoKICAgICAgICBpZiBzdGFydCBpbiBzZWVuOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHN0YWNrLCBjb21wID0gW3N0YXJ0XSwgW10KICAgICAgICBzZWVuLmFkZChzdGFydCkKICAgICAgICB3aGlsZSBzdGFjazoKICAgICAgICAgICAgYyA9IHN0YWNrLnBvcCgpOyBjb21wLmV4dGVuZChjZWxsW2NdKQogICAgICAgICAgICBmb3IgZCBpbiBuZWlnaDoKICAgICAgICAgICAgICAgIG5iID0gKGNbMF0gKyBkWzBdLCBjWzFdICsgZFsxXSwgY1syXSArIGRbMl0pCiAgICAgICAgICAgICAgICBpZiBuYiBpbiBjZWxsIGFuZCBuYiBub3QgaW4gc2VlbjoKICAgICAgICAgICAgICAgICAgICBzZWVuLmFkZChuYik7IHN0YWNrLmFwcGVuZChuYikKICAgICAgICBpZiBsZW4oY29tcCkgPiBsZW4oYmVzdCk6CiAgICAgICAgICAgIGJlc3QgPSBjb21wCiAgICByZXR1cm4gbnAuYXJyYXkoYmVzdCkgaWYgbGVuKGJlc3QpID49IG1pbl9wdHMgZWxzZSBucC5hcmFuZ2UobGVuKFApKQoKCmRlZiBfd2FsbF9kZXB0aChycik6CiAgICAiIiJEZXB0aCAoY20sIGJlbG93IHJpbSkgb2YgdGhlIGtpbG4gd2FsbC9mbG9vciBhdCByYWRpdXMgcnIgKHZlY3RvcmlzZWQpLiIiIgogICAgcmV0dXJuIG5wLndoZXJlKHJyIDw9IFJCX0NNLCBIX0NNLCAoUl9DTSAtIHJyKSAvIChSX0NNIC0gUkJfQ00pICogSF9DTSkKCgpkZWYgZXN0aW1hdGVfcG9pbnRzKFAsIHJpbV9yYWRpdXNfY209Ul9DTSwgdmlld3NfcG5nPU5vbmUsIGRlYnVnPUZhbHNlKToKICAgIFAgPSBucC5hc2FycmF5KFAsIGZsb2F0KQogICAgUCA9IFBbbnAuaXNmaW5pdGUoUCkuYWxsKDEpXQogICAgbWVkID0gbnAubWVkaWFuKFAsIDApOyBkID0gbnAubGluYWxnLm5vcm0oUCAtIG1lZCwgYXhpcz0xKQogICAgUCA9IFBbZCA8IG5wLnBlcmNlbnRpbGUoZCwgOTgpXQogICAgZGlhZyA9IGZsb2F0KG5wLmxpbmFsZy5ub3JtKFAubWF4KDApIC0gUC5taW4oMCkpKQogICAgIyBzY2FsZS1mcmVlIGxvY2FsIHBvaW50IHNwYWNpbmcgKHJvYnVzdCB0byBhIGh1Z2UgZ3JvdW5kIHBsYW5lIGluIHRoZSBzY2VuZSkKICAgIHJuZyA9IG5wLnJhbmRvbS5kZWZhdWx0X3JuZygwKQogICAgc3ViID0gUFtybmcuY2hvaWNlKGxlbihQKSwgbWluKGxlbihQKSwgNDAwMCksIHJlcGxhY2U9RmFsc2UpXQogICAgc3BhY2luZyA9IGZsb2F0KG5wLm1lZGlhbihjS0RUcmVlKFApLnF1ZXJ5KHN1Yiwgaz0yKVswXVs6LCAxXSkpCgogICAgIyAxKSB1cCBkaXJlY3Rpb24gZnJvbSB0aGUgZG9taW5hbnQgcGxhbmUgKGdyb3VuZCBvciBiaW9jaGFyIHN1cmZhY2UgLT4gc2FtZSBub3JtYWwpCiAgICBuLCBfID0gX3NlZ21lbnRfcGxhbmUoUCwgdGhyPW1heCgyLjUgKiBzcGFjaW5nLCAwLjAwMyAqIGRpYWcpKQogICAgUjEgPSByb3RfZnJvbV90byhuLCBucC5hcnJheShbMCwgMCwgMS4wXSkpCiAgICBRID0gUCBAIFIxLlQKICAgIHogPSBRWzosIDJdOyB6ciA9IHoubWF4KCkgLSB6Lm1pbigpCgogICAgIyAyKSB3aWRlc3QgaG9yaXpvbnRhbCBzbGFiID0gZ3JvdW5kOyBlbnN1cmUgaXQgc2l0cyBhdCB0aGUgYm90dG9tCiAgICBuYiwgYmVzdF93LCBncm91bmRfeiA9IDMwLCAtMSwgTm9uZQogICAgZWRnZXMgPSBucC5saW5zcGFjZSh6Lm1pbigpLCB6Lm1heCgpLCBuYiArIDEpCiAgICBmb3IgaSBpbiByYW5nZShuYik6CiAgICAgICAgbSA9ICh6ID49IGVkZ2VzW2ldKSAmICh6IDwgZWRnZXNbaSArIDFdKQogICAgICAgIGlmIG0uc3VtKCkgPCBtYXgoNTAsIDAuMDA0ICogbGVuKHopKToKICAgICAgICAgICAgY29udGludWUKICAgICAgICBjID0gUVttLCA6Ml0ubWVhbigwKQogICAgICAgIHcgPSBucC5wZXJjZW50aWxlKG5wLmh5cG90KFFbbSwgMF0gLSBjWzBdLCBRW20sIDFdIC0gY1sxXSksIDg1KQogICAgICAgIGlmIHcgPiBiZXN0X3c6CiAgICAgICAgICAgIGJlc3RfdywgZ3JvdW5kX3ogPSB3LCAwLjUgKiAoZWRnZXNbaV0gKyBlZGdlc1tpICsgMV0pCiAgICBpZiBncm91bmRfeiBpcyBOb25lOgogICAgICAgIGdyb3VuZF96ID0gei5taW4oKQogICAgZWxpZiBncm91bmRfeiA+IDAuNSAqICh6Lm1pbigpICsgei5tYXgoKSk6CiAgICAgICAgUjEgPSBucC5kaWFnKFsxLjAsIC0xLjAsIC0xLjBdKSBAIFIxICAgICAgICAgICMgZmxpcCAxODAgZGVnIGFib3V0IFgKICAgICAgICBRID0gUCBAIFIxLlQ7IHogPSBRWzosIDJdOyBncm91bmRfeiA9IC1ncm91bmRfegoKICAgICMgMykgZHJvcCB0aGUgZ3JvdW5kIHNoZWV0LCBrZWVwIHRoZSBsYXJnZXN0IGNsdXN0ZXIgKHRoZSBraWxuKQogICAga2lsbiA9IFFbeiA+IGdyb3VuZF96ICsgbWF4KDMgKiBzcGFjaW5nLCAwLjAyICogenIpXQogICAgaWR4ID0gX2xhcmdlc3RfY2x1c3RlcihraWxuLCBlcHM9My4wICogc3BhY2luZykKICAgIEsgPSBraWxuW2lkeF0KCiAgICAjIDNiKSByZWZpbmUgdGhlIGF4aXM6IHRoZSBraWxuIGlzIGEgc3VyZmFjZSBvZiByZXZvbHV0aW9uLCBzbyBpdHMgc3ltbWV0cnkKICAgICMgYXhpcyBpcyB0aGUgc21hbGxlc3QtdmFyaWFuY2UgUENBIGRpcmVjdGlvbiAocm9idXN0IHZzIGEgdGlsdGVkIHBsYW5lIGZpdCkuCiAgICBjMCA9IEsubWVhbigwKQogICAgXywgXywgdnQgPSBucC5saW5hbGcuc3ZkKEsgLSBjMCwgZnVsbF9tYXRyaWNlcz1GYWxzZSkKICAgIGF4aXMgPSB2dFsyXQogICAgaWYgYXhpcyBAIG5wLmFycmF5KFswLCAwLCAxLjBdKSA8IDA6CiAgICAgICAgYXhpcyA9IC1heGlzCiAgICBLID0gKEsgLSBjMCkgQCByb3RfZnJvbV90byhheGlzLCBucC5hcnJheShbMCwgMCwgMS4wXSkpLlQKICAgICMgcmltICh3aWRlIGVuZCkgbXVzdCBiZSBhdCArWjogcmFkaXVzIHNob3VsZCBncm93IHdpdGggaGVpZ2h0CiAgICByaG8gPSBucC5oeXBvdChLWzosIDBdLCBLWzosIDFdKQogICAgaWYgbnAuY29ycmNvZWYoS1s6LCAyXSwgcmhvKVswLCAxXSA8IDA6CiAgICAgICAgS1s6LCAyXSAqPSAtMS4wCiAgICBpZiBkZWJ1ZzoKICAgICAgICBwcmludChmIiAgW2RlYnVnXSBzcGFjaW5nPXtzcGFjaW5nOi4zZn0gbl9raWxuPXtsZW4oSyl9L3tsZW4oa2lsbil9IGF4aXNfej17YXhpc1syXTouM2Z9IikKCiAgICAjIDQpIGZpdCB0aGUga2lsbiBXQUxMIGNvbmUgLT4gcmltIHJhZGl1cyAmIHBsYW5lIC0+IHNjYWxlIHRvIGNtIChheGlzIGF0IG9yaWdpbikuCiAgICAjIFRoZSB3YWxsJ3MgbWF4LXJhZGl1cy12cy1oZWlnaHQgaXMgYSBzdHJhaWdodCBsaW5lOyBleHRyYXBvbGF0ZSB0byB0aGUgdG9wLgogICAgemsgPSBLWzosIDJdCiAgICByaG8gPSBucC5oeXBvdChLWzosIDBdLCBLWzosIDFdKQogICAgemIgPSBucC5saW5zcGFjZSh6ay5taW4oKSwgemsubWF4KCksIDIyKQogICAgenosIHJyID0gW10sIFtdCiAgICBmb3IgaSBpbiByYW5nZShsZW4oemIpIC0gMSk6CiAgICAgICAgbSA9ICh6ayA+PSB6YltpXSkgJiAoemsgPCB6YltpICsgMV0pCiAgICAgICAgaWYgbS5zdW0oKSA8IDIwOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIHp6LmFwcGVuZCgwLjUgKiAoemJbaV0gKyB6YltpICsgMV0pKTsgcnIuYXBwZW5kKG5wLnBlcmNlbnRpbGUocmhvW21dLCA5OCkpCiAgICB6eiwgcnIgPSBucC5hcnJheSh6eiksIG5wLmFycmF5KHJyKQogICAgbV9zbG9wZSwgY19pbnQgPSBucC5saW5hbGcubHN0c3EobnAuY19benosIG5wLm9uZXNfbGlrZSh6eildLCByciwgcmNvbmQ9Tm9uZSlbMF0KICAgIHpfcmltID0gZmxvYXQobnAucGVyY2VudGlsZSh6aywgOTkuNSkpCiAgICByX3VuaXRzID0gbV9zbG9wZSAqIHpfcmltICsgY19pbnQKICAgIHMgPSByaW1fcmFkaXVzX2NtIC8gcl91bml0cwogICAgaWYgZGVidWc6CiAgICAgICAgcHJpbnQoZiIgIFtkZWJ1Z10gc2xvcGU9e21fc2xvcGU6LjNmfSByX3VuaXRzPXtyX3VuaXRzOi4zZn0gcz17czouNGZ9IikKICAgIEsgPSAoSyAtIG5wLmFycmF5KFswLjAsIDAuMCwgel9yaW1dKSkgKiBzCgogICAgIyA1KSBUT1Atc3VyZmFjZSBoZWlnaHRtYXAgb3ZlciB0aGUgcmltIGRpc2ssIGludGVncmF0ZWQgYWdhaW5zdCB0aGUga25vd24gY29uZS4KICAgICMgICAgUGVyIGNlbGwgdGFrZSB0aGUgU0hBTExPV0VTVCBwb2ludHMgKHRoZSBiaW9jaGFyIHRvcCkgLT4gaWdub3JlcyBkZWVwCiAgICAjICAgIGludGVyaW9yIC8gcmVjb25zdHJ1Y3Rpb24gYXJ0ZWZhY3RzLCBhbmQgd29ya3MgZm9yIGFueSBmaWxsIGxldmVsLgogICAgZnJvbSBjb2xsZWN0aW9ucyBpbXBvcnQgZGVmYXVsdGRpY3QKICAgIHgsIHksIHpjID0gS1s6LCAwXSwgS1s6LCAxXSwgS1s6LCAyXQogICAgZGVwID0gLXpjOyByaG8gPSBucC5oeXBvdCh4LCB5KQogICAgVl9mdWxsID0gKDEgLyAzKSAqIG5wLnBpICogSF9DTSAqIChSQl9DTSAqKiAyICsgUkJfQ00gKiBSX0NNICsgUl9DTSAqKiAyKSAvIDEwMDAuMAogICAgY29sZ3JpZCA9IE5vbmUgICAgICAgICAgICAgICAgICAgICMgYmlvY2hhci1kZXB0aCBoZWF0bWFwIChmaWxsZWQgaW4gYmVsb3cpCgogICAgaW5zID0gcmhvIDw9IFJfQ00KICAgIGd4ID0gbnAuZmxvb3IoKHhbaW5zXSArIFJfQ00pIC8gQ0VMTCkuYXN0eXBlKGludCkKICAgIGd5ID0gbnAuZmxvb3IoKHlbaW5zXSArIFJfQ00pIC8gQ0VMTCkuYXN0eXBlKGludCkKICAgIGRlcGkgPSBkZXBbaW5zXQogICAgYWNjID0gZGVmYXVsdGRpY3QobGlzdCkKICAgIGZvciB4aSwgeWksIGRwIGluIHppcChneCwgZ3ksIGRlcGkpOgogICAgICAgIGFjY1soeGksIHlpKV0uYXBwZW5kKGRwKQogICAgY2VsbHMsIGRlcHRocyA9IFtdLCBbXQogICAgZm9yIGtleSwgdiBpbiBhY2MuaXRlbXMoKToKICAgICAgICBpZiBsZW4odikgPCAzOgogICAgICAgICAgICBjb250aW51ZQogICAgICAgIGNlbGxzLmFwcGVuZChrZXkpOyBkZXB0aHMuYXBwZW5kKG5wLnBlcmNlbnRpbGUodiwgVE9QX1BDVCkpICAgIyB0b3AgPSBiaW9jaGFyIHN1cmZhY2UKICAgIGlmIGxlbihjZWxscykgPCAzMDogICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgIyBlc3NlbnRpYWxseSBlbXB0eSBraWxuCiAgICAgICAgVl9MID0gVl9zaW1wbGUgPSBoX2ZpbGwgPSAwLjAKICAgIGVsc2U6CiAgICAgICAgY2VsbHMgPSBucC5hcnJheShjZWxscyk7IGRlcHRocyA9IG5wLmFycmF5KGRlcHRocykKICAgICAgICBjZW50ZXJzID0gKGNlbGxzICsgMC41KSAqIENFTEwgLSBSX0NNCiAgICAgICAgaW50ZXJwID0gTmVhcmVzdE5ESW50ZXJwb2xhdG9yKGNlbnRlcnMsIGRlcHRocykKICAgICAgICBuY2VsbCA9IGludChucC5jZWlsKDIgKiBSX0NNIC8gQ0VMTCkpCiAgICAgICAgY2MgPSAobnAuYXJhbmdlKG5jZWxsKSArIDAuNSkgKiBDRUxMIC0gUl9DTQogICAgICAgIFhYLCBZWSA9IG5wLm1lc2hncmlkKGNjLCBjYyk7IFJSID0gbnAuaHlwb3QoWFgsIFlZKQogICAgICAgIGRpc2sgPSBSUiA8PSBSX0NNCiAgICAgICAgZHN1cmYgPSBpbnRlcnAoWFhbZGlza10sIFlZW2Rpc2tdKQogICAgICAgIGNvbCA9IF93YWxsX2RlcHRoKFJSW2Rpc2tdKSAtIGRzdXJmCiAgICAgICAgVl9MID0gZmxvYXQoY29sW2NvbCA+IENPTF9NSU5dLnN1bSgpICogQ0VMTCAqIENFTEwgLyAxMDAwLjApCiAgICAgICAgY2cgPSBfd2FsbF9kZXB0aChSUikgLSBpbnRlcnAoWFgsIFlZKSAgICAgICAgICAjIGZ1bGwtZ3JpZCBiaW9jaGFyIGRlcHRoIGhlYXRtYXAKICAgICAgICBjb2xncmlkID0gbnAud2hlcmUoZGlzayAmIChjZyA+IENPTF9NSU4pLCBjZywgbnAubmFuKQogICAgICAgICMgZmxhdCBjcm9zcy1jaGVjayBmcm9tIHRoZSBjZWxscyB0aGF0IGFjdHVhbGx5IGhvbGQgYmlvY2hhcgogICAgICAgIGNvbGMgPSBfd2FsbF9kZXB0aChucC5oeXBvdChjZW50ZXJzWzosIDBdLCBjZW50ZXJzWzosIDFdKSkgLSBkZXB0aHMKICAgICAgICBiaW9fZCA9IGRlcHRoc1tjb2xjID4gQ09MX01JTl0KICAgICAgICBkX21lZCA9IGZsb2F0KG5wLm1lZGlhbihiaW9fZCkpIGlmIGxlbihiaW9fZCkgZWxzZSBmbG9hdChucC5tZWRpYW4oZGVwdGhzKSkKICAgICAgICBoX2ZpbGwgPSBmbG9hdChucC5jbGlwKEhfQ00gLSBkX21lZCwgMCwgSF9DTSkpCiAgICAgICAgcnMgPSBSQl9DTSArIChSX0NNIC0gUkJfQ00pICogKGhfZmlsbCAvIEhfQ00pCiAgICAgICAgVl9zaW1wbGUgPSAoMSAvIDMpICogbnAucGkgKiBoX2ZpbGwgKiAoUkJfQ00gKiogMiArIFJCX0NNICogcnMgKyBycyAqKiAyKSAvIDEwMDAuMAogICAgICAgIGlmIGRlYnVnOgogICAgICAgICAgICBwID0gbnAucGVyY2VudGlsZShkZXB0aHMsIFsxMCwgNTAsIDkwXSkKICAgICAgICAgICAgcHJpbnQoZiIgIFtkZWJ1Z10gY2VsbHM9e2xlbihkZXB0aHMpfSB0b3BfZGVwdGggcDEwLzUwLzkwPSIKICAgICAgICAgICAgICAgICAgZiJ7cFswXTouMGZ9L3twWzFdOi4wZn0ve3BbMl06LjBmfSBWPXtWX0w6LjBmfSBWZmxhdD17Vl9zaW1wbGU6LjBmfSIpCgogICAgaWYgdmlld3NfcG5nOgogICAgICAgIGltcG9ydCBtYXRwbG90bGliOyBtYXRwbG90bGliLnVzZSgiQWdnIik7IGltcG9ydCBtYXRwbG90bGliLnB5cGxvdCBhcyBwbHQKICAgICAgICBmcm9tIG1hdHBsb3RsaWIucGF0Y2hlcyBpbXBvcnQgQ2lyY2xlCiAgICAgICAgZGVwQSA9IC1LWzosIDJdOyByaG9BID0gbnAuaHlwb3QoS1s6LCAwXSwgS1s6LCAxXSkKICAgICAgICBjb2xBID0gX3dhbGxfZGVwdGgobnAuY2xpcChyaG9BLCAwLCBSX0NNKSkgLSBkZXBBICAgICAgICAgICMgYmlvY2hhciBiZW5lYXRoIGVhY2ggcHQKICAgICAgICBpc19iaW8gPSAocmhvQSA8PSBSX0NNKSAmIChjb2xBID4gQ09MX01JTikgICAgICAgICAgICAgICAgICMgYmlvY2hhciB2cyBraWxuIHN0cnVjdHVyZQogICAgICAgIGZpZywgYXggPSBwbHQuc3VicGxvdHMoMSwgMywgZmlnc2l6ZT0oMTYsIDUuMikpCiAgICAgICAgIyAxKSBUT1A6IGtpbG4gKGdyZXkpIHdpdGggYmlvY2hhciBjb2xvdXJlZCBieSBob3cgZGVlcCB0aGUgYmlvY2hhciBpcwogICAgICAgIGF4WzBdLnNjYXR0ZXIoS1t+aXNfYmlvLCAwXSwgS1t+aXNfYmlvLCAxXSwgcz0xLCBjPSIjY2ZjN2I2IiwgbGluZXdpZHRocz0wKQogICAgICAgIGlmIGlzX2Jpby5hbnkoKToKICAgICAgICAgICAgYXhbMF0uc2NhdHRlcihLW2lzX2JpbywgMF0sIEtbaXNfYmlvLCAxXSwgcz00LCBjPWNvbEFbaXNfYmlvXSwgY21hcD0iaW5mZXJubyIsIGxpbmV3aWR0aHM9MCkKICAgICAgICBheFswXS5hZGRfcGF0Y2goQ2lyY2xlKCgwLCAwKSwgUl9DTSwgZmlsbD1GYWxzZSwgZWM9IiNBOTRFMjgiLCBsdz0yKSkKICAgICAgICBheFswXS5zZXRfdGl0bGUoIlRPUCDigJQgYmlvY2hhciAoY29sb3VyKSBpbnNpZGUgdGhlIGtpbG4gKGdyZXkpIikKICAgICAgICAjIDIpIFNJREU6IGJpb2NoYXIgc2l0dGluZyBpbnNpZGUgdGhlIEtvbi1UaWtpIGNvbmUKICAgICAgICBheFsxXS5zY2F0dGVyKEtbfmlzX2JpbywgMF0sIEtbfmlzX2JpbywgMl0sIHM9MSwgYz0iI2NmYzdiNiIsIGxpbmV3aWR0aHM9MCkKICAgICAgICBpZiBpc19iaW8uYW55KCk6CiAgICAgICAgICAgIGF4WzFdLnNjYXR0ZXIoS1tpc19iaW8sIDBdLCBLW2lzX2JpbywgMl0sIHM9NCwgYz1jb2xBW2lzX2Jpb10sIGNtYXA9ImluZmVybm8iLCBsaW5ld2lkdGhzPTApCiAgICAgICAgYXhbMV0uc2V0X3RpdGxlKCJTSURFIOKAlCBiaW9jaGFyIHNpdHMgaW5zaWRlIHRoZSBLb24tVGlraSBjb25lIikKICAgICAgICAjIDMpIGJpb2NoYXItZGVwdGggaGVhdG1hcCA9IHRoZSB2b2x1bWUgaW50ZWdyYW5kCiAgICAgICAgaWYgY29sZ3JpZCBpcyBub3QgTm9uZToKICAgICAgICAgICAgaW0gPSBheFsyXS5pbXNob3coY29sZ3JpZCwgb3JpZ2luPSJsb3dlciIsIGV4dGVudD1bLVJfQ00sIFJfQ00sIC1SX0NNLCBSX0NNXSwKICAgICAgICAgICAgICAgICAgICAgICAgICAgICAgY21hcD0iaW5mZXJubyIsIGludGVycG9sYXRpb249Im5lYXJlc3QiKQogICAgICAgICAgICBmaWcuY29sb3JiYXIoaW0sIGF4PWF4WzJdLCBmcmFjdGlvbj0wLjA0NiwgcGFkPTAuMDQsIGxhYmVsPSJiaW9jaGFyIGRlcHRoIChjbSkiKQogICAgICAgICAgICBheFsyXS5hZGRfcGF0Y2goQ2lyY2xlKCgwLCAwKSwgUl9DTSwgZmlsbD1GYWxzZSwgZWM9IiNBOTRFMjgiLCBsdz0xLjUpKQogICAgICAgIGF4WzJdLnNldF90aXRsZSgiQmlvY2hhciBoZWF0bWFwIOKAlCBkZXB0aCBwZXIgc3BvdCAodm9sdW1lID0gc3VtKSIpCiAgICAgICAgZm9yIGEgaW4gYXg6CiAgICAgICAgICAgIGEuc2V0X2FzcGVjdCgiZXF1YWwiLCAiYm94IikKICAgICAgICBmaWcudGlnaHRfbGF5b3V0KCk7IGZpZy5zYXZlZmlnKHZpZXdzX3BuZywgZHBpPTEyMCk7IHBsdC5jbG9zZShmaWcpCgogICAgcmV0dXJuIHsidm9sdW1lX0wiOiBWX0wsICJ2b2x1bWVfTF9mbGF0ZmlsbCI6IFZfc2ltcGxlLCAiZmlsbF9oZWlnaHRfY20iOiBoX2ZpbGwsCiAgICAgICAgICAgICJmaWxsX3BjdCI6IDEwMCAqIFZfTCAvIFZfZnVsbCwgIndlaWdodF9rZyI6IERFTlNJVFkgKiBWX0wsCiAgICAgICAgICAgICJtZWFzdXJlZF9yaW1fdW5pdHMiOiByX3VuaXRzLCAic2NhbGVfY21fcGVyX3VuaXQiOiBzfQoKCmRlZiBlc3RpbWF0ZShwbHlfcGF0aCwgcmltX3JhZGl1c19jbT1SX0NNLCB2aWV3c19wbmc9Tm9uZSk6CiAgICBpbXBvcnQgb3BlbjNkIGFzIG8zZAogICAgcGNkID0gbzNkLmlvLnJlYWRfcG9pbnRfY2xvdWQocGx5X3BhdGgpCiAgICBpZiBsZW4ocGNkLnBvaW50cykgPT0gMDoKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KCJlbXB0eSBwb2ludCBjbG91ZCIpCiAgICByZXR1cm4gZXN0aW1hdGVfcG9pbnRzKG5wLmFzYXJyYXkocGNkLnBvaW50cyksIHJpbV9yYWRpdXNfY20sIHZpZXdzX3BuZykKCgpkZWYgX3ByaW50KHJlcyk6CiAgICBwcmludCgiPSIgKiA0NikKICAgIHByaW50KGYiICBCSU9DSEFSIFZPTFVNRSAoaW50ZWdyYXRlZCkgOiB7cmVzWyd2b2x1bWVfTCddOjYuMGZ9IEwiKQogICAgcHJpbnQoZiIgIGNyb3NzLWNoZWNrIChmbGF0IGZpbGwpICAgICA6IHtyZXNbJ3ZvbHVtZV9MX2ZsYXRmaWxsJ106Ni4wZn0gTCIpCiAgICBwcmludChmIiAgZmlsbCBoZWlnaHQgLyBmaWxsICUgICAgICAgIDoge3Jlc1snZmlsbF9oZWlnaHRfY20nXTouMGZ9IGNtIC8ge3Jlc1snZmlsbF9wY3QnXTouMGZ9JSIpCiAgICBwcmludChmIiAgYXBwcm94IHdlaWdodCAofjAuMjUga2cvTCkgIDoge3Jlc1snd2VpZ2h0X2tnJ106Ni4wZn0ga2ciKQogICAgcHJpbnQoZiIgIHNjYWxlICAgICAgICAgICAgICAgICAgICAgICA6IHtyZXNbJ3NjYWxlX2NtX3Blcl91bml0J106LjRmfSBjbS91bml0IikKICAgIHByaW50KCI9IiAqIDQ2KQoKCmlmIF9fbmFtZV9fID09ICJfX21haW5fXyI6CiAgICBpZiBsZW4oc3lzLmFyZ3YpIDwgMjoKICAgICAgICByYWlzZSBTeXN0ZW1FeGl0KF9fZG9jX18pCiAgICBwbHkgPSBzeXMuYXJndlsxXQogICAgcmltID0gZmxvYXQoc3lzLmFyZ3ZbMl0pIGlmIGxlbihzeXMuYXJndikgPiAyIGVsc2UgUl9DTQogICAgcG5nID0gc3lzLmFyZ3ZbM10gaWYgbGVuKHN5cy5hcmd2KSA+IDMgZWxzZSBOb25lCiAgICBfcHJpbnQoZXN0aW1hdGUocGx5LCByaW0sIHBuZykpCg==").decode("utf-8"))
from estimate_volume import estimate_points
from vggt.models.vggt import VGGT
from vggt.utils.load_fn import load_and_preprocess_images
from vggt.utils.pose_enc import pose_encoding_to_extri_intri
from vggt.utils.geometry import unproject_depth_map_to_point_map

MODEL = VGGT.from_pretrained("facebook/VGGT-1B").to("cuda").eval()
import torchvision
print("Setup ready | GPU:", torch.cuda.get_device_name(0),
      "| torch", torch.__version__, "| torchvision", torchvision.__version__)

def extract_frames(video, n=32, out="frames"):
    if os.path.exists(out): shutil.rmtree(out)
    os.makedirs(out)
    cap = cv2.VideoCapture(video); total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT) or 0)
    if total <= 0:
        total = 0
        while cap.grab(): total += 1
        cap.set(cv2.CAP_PROP_POS_FRAMES, 0)
    slot = total / n; k = 0
    for i in range(n):
        lo, hi = int(i*slot), int((i+1)*slot); best = None
        for idx in np.linspace(lo, max(lo, hi-1), 5).astype(int):
            cap.set(cv2.CAP_PROP_POS_FRAMES, int(idx)); ok, fr = cap.read()
            if not ok: continue
            s = cv2.Laplacian(cv2.cvtColor(fr, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()
            if best is None or s > best[1]: best = (idx, s, fr)
        if best:
            cv2.imwrite(f"{out}/f_{k:03d}.jpg", best[2], [cv2.IMWRITE_JPEG_QUALITY, 95]); k += 1
    cap.release(); return sorted(glob.glob(f"{out}/*.jpg"))

def reconstruct(paths):
    dt = torch.bfloat16 if torch.cuda.get_device_capability()[0] >= 8 else torch.float16
    imgs = load_and_preprocess_images(paths).to("cuda")
    with torch.no_grad(), torch.cuda.amp.autocast(dtype=dt):
        pred = MODEL(imgs)
    def gk(d, *ks):
        for kk in ks:
            if kk in d: return d[kk]
        raise KeyError(ks)
    extr, intr = pose_encoding_to_extri_intri(gk(pred, "pose_enc"), imgs.shape[-2:])
    depth = gk(pred, "depth", "depth_map"); conf = gk(pred, "depth_conf", "point_conf", "depth_confidence")
    w = np.asarray(unproject_depth_map_to_point_map(depth.squeeze(0), extr.squeeze(0), intr.squeeze(0))).reshape(-1, 3)
    conf = conf.squeeze(0).float().cpu().numpy().reshape(-1)
    w = w[(conf >= np.quantile(conf, 0.5)) & np.isfinite(w).all(1)]
    if len(w) > 300000:
        w = w[np.random.default_rng(0).choice(len(w), 300000, replace=False)]
    return w

def save_ply(path, P):
    P = np.asarray(P, np.float32)
    hdr = ("ply\nformat binary_little_endian 1.0\n"
           f"element vertex {len(P)}\n"
           "property float x\nproperty float y\nproperty float z\nend_header\n")
    with open(path, "wb") as f:
        f.write(hdr.encode()); f.write(P.tobytes())
print("helpers ready: extract_frames(), reconstruct(), estimate_points()")

### Cell 2 · Measure a kiln — upload a video, get the volume
Run this cell, pick your slow-orbit video, and wait. It extracts frames, builds the
3-D model, and prints the biochar volume. Re-run it for each new kiln.

In [ ]:
from google.colab import files
from IPython.display import Image, display

up = files.upload()                       # pick your slow-orbit .mp4/.mov
VIDEO = list(up.keys())[0]
print("video:", VIDEO)

print("1/3 extracting frames..."); paths = extract_frames(VIDEO); print("   ", len(paths), "frames")
print("2/3 reconstructing 3-D (GPU)..."); world = reconstruct(paths); print("   ", len(world), "points")
print("3/3 measuring volume...")
save_ply("dense.ply", world)
res = estimate_points(world, rim_radius_cm=75.0, views_png="kiln_views.png")

print("\n" + "=" * 46)
print(f"  BIOCHAR VOLUME : {res['volume_L']:6.0f} L    (~{res['weight_kg']:.0f} kg)")
print(f"  fill level     : {res['fill_height_cm']:.0f} cm  ({res['fill_pct']:.0f}% of a ~1000 L kiln)")
print(f"  cross-check    : {res['volume_L_flatfill']:6.0f} L   (should be close to the volume)")
print("=" * 46)
gap = abs(res['volume_L'] - res['volume_L_flatfill']) / max(res['volume_L'], 1) * 100
print(("OK: the two estimates agree (%.0f%%)." % gap) if gap < 8 else
      ("WARNING: estimates disagree by %.0f%% -> cloud is noisy, re-shoot per the SOP." % gap))
display(Image("kiln_views.png"))          # TOP should be a disk, SIDE a cone/bowl

### Notes
- **Rim scale:** assumes a standard Kon-Tiki 1000 rim (Ø150 cm). For other kilns, change
  `rim_radius_cm`, or lay a 1-metre marker in the video.
- **Trust check:** if the two volume numbers disagree by more than ~8%, or the 3-D views
  don't look like a clean bowl, the capture was poor — re-shoot per the SOP.
- **Downloads:** `dense.ply` (3-D model) and `kiln_views.png` are saved in the file browser.
